# Tutorial: Information Theory Basics

> **Type:** Tutorial. Follow along step by step. Your only job is to make it work.
>
> **Time:** ~1.5 hours
> **Prerequisites:** Module 03 (probability and distributions)
> **What you'll build:** Entropy, cross-entropy, and KL divergence implemented from scratch in numpy. You'll also see why cross-entropy punishes confident wrong predictions, watch entropy and KL divergence move as a distribution changes, and compare feature rankings where mutual information catches a non-linear relationship that correlation misses.

---

## What We're Building

Four short parts:
1. Implement entropy, cross-entropy, and KL divergence from scratch, and verify the identity H(P, Q) = H(P) + KL(P || Q)
2. Show that cross-entropy loss penalises confident wrong predictions far more than uncertain wrong ones
3. Visualise entropy and KL divergence as a distribution's parameter changes
4. Use mutual information to rank features on a dataset with a non-linear relationship, and compare the ranking with plain correlation

Don't worry about deriving anything new here. `explanation.md` already covered that. This notebook is about seeing the four concepts behave the way the theory predicts.

## Step 0: Setup

In [ ]:
# !pip install numpy scipy matplotlib scikit-learn pandas

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif
import pandas as pd

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
print('Setup complete.')

---

## Part 1: Entropy, Cross-Entropy, and KL Divergence From Scratch

### Step 1: Entropy

Entropy measures how uncertain a distribution is. We'll compute it for three distributions over a 4-sided die: a fair one, a biased one, and a point mass (fully certain).

In [ ]:
def entropy(p, eps=1e-12):
    """Shannon entropy in bits. p is a 1D array of probabilities that sums to 1."""
    p = np.asarray(p, dtype=float)
    p = np.clip(p, eps, 1.0)  # avoid log(0)
    return -np.sum(p * np.log2(p))

fair_die     = np.array([0.25, 0.25, 0.25, 0.25])
biased_die   = np.array([0.70, 0.10, 0.10, 0.10])
point_mass   = np.array([1.00, 0.00, 0.00, 0.00])

for name, dist in [('Fair die', fair_die), ('Biased die', biased_die), ('Point mass', point_mass)]:
    print(f"{name:12s}: H = {entropy(dist):.4f} bits")

print(f"\nMax possible entropy for 4 outcomes: log2(4) = {np.log2(4):.4f} bits (matches the fair die)")

> **What just happened?** The fair die has maximum entropy: 2 bits, because you need 2 bits to identify one of 4 equally likely outcomes. The biased die has less uncertainty, because one outcome dominates. The point mass has zero entropy. There's nothing to be uncertain about.

### Step 2: Cross-Entropy as a Loss Function

Cross-entropy measures the cost of using predicted distribution Q when the truth is P. We'll simulate a binary classifier making predictions of varying confidence and quality, and watch how the loss responds.

In [ ]:
def binary_cross_entropy(y_true, y_pred, eps=1e-12):
    """Binary cross-entropy loss, averaged over examples. y_true in {0,1}, y_pred in (0,1)."""
    y_pred = np.clip(y_pred, eps, 1 - eps)  # avoid log(0)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# A single true label, y = 1, with predictions ranging from very wrong to very right
y_true_single = np.array([1.0])
predictions = np.linspace(0.01, 0.99, 200)

losses = [binary_cross_entropy(y_true_single, np.array([p])) for p in predictions]

plt.figure(figsize=(8, 4.5))
plt.plot(predictions, losses, color='#E15759', linewidth=2)
plt.axvline(1.0, color='gray', linestyle='--', alpha=0.5, label='True label (y=1)')
plt.xlabel('Predicted probability ŷ that y=1')
plt.ylabel('Binary cross-entropy loss')
plt.title('Cross-entropy loss vs. predicted probability (true label y=1)')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Confident and correct   (ŷ=0.99): loss = {binary_cross_entropy(y_true_single, np.array([0.99])):.4f}")
print(f"Uncertain               (ŷ=0.50): loss = {binary_cross_entropy(y_true_single, np.array([0.50])):.4f}")
print(f"Confident and WRONG     (ŷ=0.01): loss = {binary_cross_entropy(y_true_single, np.array([0.01])):.4f}")

> **What just happened?** The loss is low and nearly flat when the prediction is confidently correct, but explodes as the prediction becomes confidently wrong. A confidently wrong prediction (ŷ=0.01 when y=1) costs about 6.6 times as much as an uncertain one (ŷ=0.5), and about 460 times as much as a confidently correct one (ŷ=0.99). That asymmetry is why cross-entropy, not MSE, is the standard loss for classification: it makes the model pay heavily for overconfident mistakes.

### Step 3: KL Divergence and the H(P,Q) = H(P) + KL(P||Q) Identity

KL divergence measures how far a distribution Q is from a reference distribution P. We'll implement it, confirm it's asymmetric, and verify the identity linking it to entropy and cross-entropy.

In [ ]:
def cross_entropy_dist(p, q, eps=1e-12):
    """Cross-entropy H(P, Q) between two discrete distributions, in bits."""
    p = np.asarray(p, dtype=float)
    q = np.clip(np.asarray(q, dtype=float), eps, 1.0)
    return -np.sum(p * np.log2(q))

def kl_divergence(p, q, eps=1e-12):
    """KL(P || Q) in bits. p and q are discrete distributions (sum to 1)."""
    p = np.clip(np.asarray(p, dtype=float), eps, 1.0)
    q = np.clip(np.asarray(q, dtype=float), eps, 1.0)
    return np.sum(p * np.log2(p / q))

P = np.array([0.50, 0.30, 0.15, 0.05])
Q = np.array([0.30, 0.30, 0.20, 0.20])

kl_pq = kl_divergence(P, Q)
kl_qp = kl_divergence(Q, P)
print(f"KL(P || Q) = {kl_pq:.4f} bits")
print(f"KL(Q || P) = {kl_qp:.4f} bits")
print(f"Asymmetric: KL(P||Q) != KL(Q||P) -> {not np.isclose(kl_pq, kl_qp)}")

# Verify H(P, Q) = H(P) + KL(P || Q)
h_p = entropy(P)
h_pq = cross_entropy_dist(P, Q)
print(f"\nH(P)              = {h_p:.4f} bits")
print(f"H(P) + KL(P||Q)   = {h_p + kl_pq:.4f} bits")
print(f"H(P, Q)           = {h_pq:.4f} bits")
print(f"Identity holds: {np.isclose(h_p + kl_pq, h_pq)}")

# scipy's built-in check (scipy.stats.entropy computes KL when a second array is given)
print(f"\nscipy.stats.entropy(P, Q, base=2) = {stats.entropy(P, Q, base=2):.4f} bits (should match KL(P||Q) above)")

> **What just happened?** KL(P||Q) and KL(Q||P) came out different. That confirms KL divergence isn't a symmetric distance. It's a directed measure of how much "surprise" you'd incur by using Q when P is true. The identity H(P,Q) = H(P) + KL(P||Q) held exactly: cross-entropy is always the true entropy plus a penalty for using the wrong distribution.

---

## Part 2: Visualising Entropy and KL Divergence

### Step 4: How Entropy and KL Divergence Move as a Distribution Changes

We'll sweep a Bernoulli distribution's parameter p from 0 to 1 and plot two curves: its entropy H(p), and its KL divergence from a fixed reference distribution Bernoulli(0.5).

In [ ]:
def bernoulli_entropy(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def bernoulli_kl(p, q, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    q = np.clip(q, eps, 1 - eps)
    return p * np.log2(p / q) + (1 - p) * np.log2((1 - p) / (1 - q))

p_values = np.linspace(0.001, 0.999, 300)
entropies = bernoulli_entropy(p_values)
kl_from_half = bernoulli_kl(p_values, 0.5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(p_values, entropies, color='#4E79A7', linewidth=2)
axes[0].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel('p (probability of outcome 1)')
axes[0].set_ylabel('H(p) in bits')
axes[0].set_title('Entropy of Bernoulli(p)\n(peaks at p=0.5 — maximum uncertainty)')

axes[1].plot(p_values, kl_from_half, color='#E15759', linewidth=2)
axes[1].axvline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[1].set_xlabel('p (probability of outcome 1)')
axes[1].set_ylabel('KL(Bernoulli(p) || Bernoulli(0.5)) in bits')
axes[1].set_title('KL divergence from Bernoulli(0.5)\n(zero only at p=0.5)')

plt.tight_layout()
plt.show()

> **What just happened?** Entropy peaks at p=0.5 (maximum uncertainty) and falls to zero at the extremes (p=0 or p=1, where the outcome is certain). KL divergence from Bernoulli(0.5) does the opposite. It's zero exactly at p=0.5, where the two distributions match, and grows as p moves away in either direction. Entropy asks "how uncertain is this distribution on its own?" KL divergence asks "how far is this distribution from a reference?" They're two different questions that share the same log-probability machinery.

---

## Part 3: Mutual Information for Feature Selection

### Step 5: When Correlation Misses What Mutual Information Catches

We'll build a small dataset where one feature is linearly related to the label, which correlation will find. Another is related in a purely non-linear way. Correlation will miss it, but mutual information won't.

In [ ]:
n_samples = 1000
rng = np.random.RandomState(RANDOM_SEED)

# Feature 1: linearly related to the label
label = rng.randint(0, 2, size=n_samples)
feature_linear = label + rng.normal(0, 0.5, size=n_samples)

# Feature 2: non-linearly related to the label (label determines the *variance*, not the mean)
# Correlation with the label is close to zero even though the feature is highly informative
feature_nonlinear = np.where(
    label == 1,
    rng.normal(0, 2.0, size=n_samples),
    rng.normal(0, 0.3, size=n_samples),
)

# Feature 3: pure noise, unrelated to the label
feature_noise = rng.normal(0, 1, size=n_samples)

df = pd.DataFrame({
    'feature_linear': feature_linear,
    'feature_nonlinear': feature_nonlinear,
    'feature_noise': feature_noise,
    'label': label,
})

X = df[['feature_linear', 'feature_nonlinear', 'feature_noise']]
y = df['label']

correlations = X.apply(lambda col: np.corrcoef(col, y)[0, 1]).abs()
mi_scores = pd.Series(
    mutual_info_classif(X, y, random_state=RANDOM_SEED),
    index=X.columns,
)

comparison = pd.DataFrame({
    '|correlation|': correlations,
    'mutual_information': mi_scores,
}).sort_values('mutual_information', ascending=False)

print(comparison)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

comparison['|correlation|'].plot(kind='bar', ax=axes[0], color='#4E79A7')
axes[0].set_title('Ranking by |correlation|\n(misses the non-linear feature)')
axes[0].set_ylabel('|correlation| with label')
axes[0].tick_params(axis='x', rotation=20)

comparison['mutual_information'].plot(kind='bar', ax=axes[1], color='#59A14F')
axes[1].set_title('Ranking by mutual information\n(correctly ranks feature_nonlinear as informative)')
axes[1].set_ylabel('Mutual information (nats)')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

> **What just happened?** `feature_nonlinear` has almost no linear correlation with the label. Its *mean* doesn't depend on the label; only its *spread* does. So a correlation-based feature selector would rank it near the noise feature and discard it. Mutual information has no such blind spot. Because it's built from entropy rather than covariance, it detects that knowing the label changes the feature's distribution, and ranks it as informative. This is the failure mode `explanation.md` warned about: correlation only sees linear relationships.

---

## What You Built

You implemented entropy, cross-entropy, and KL divergence from scratch, and verified that H(P,Q) = H(P) + KL(P||Q) exactly. You saw cross-entropy loss punish a confidently wrong prediction about 6.6 times more than an uncertain one, and about 460 times more than a confidently correct one. You watched entropy and KL divergence move in opposite ways as a distribution's parameter changes. And you used mutual information to flag a non-linearly related feature that correlation missed.

## What's Next

- Reread `explanation.md` now that you've seen the four quantities behave. The connections between them should feel more concrete.
- Open `howto.md` when you need to compute entropy, cross-entropy, KL divergence, or mutual information on your own data.
- Keep `reference.md` open as a formula and API cheat sheet.
- **Extension challenge:** replace `mutual_info_classif` with `mutual_info_regression`, and build a regression version of the non-linear-feature example (for example, a feature that determines the label's *variance* rather than its class).

---

**AWS SageMaker version:** this module doesn't need a separate SageMaker notebook. Everything here runs on CPU in seconds. If you're doing feature selection on a large dataset, see `howto.md` for how to run `mutual_info_classif` inside a SageMaker Processing job instead of locally.